# 11 · Transition-metal catalysts

Whole complexes rather than the coordination sphere on its own: a catalyst written as SMILES, searched,
gated, and ranked. `07_metal` covers what the sphere itself can be told to do; this is that applied to
complete complexes.

Everything here is `rx.metal` + `rx.embed` with the arguments from the earlier notebooks. What changes is
scale: a real ligand set, a conformer search on top, and a gate deciding what to keep.


In [ ]:
import tempfile

import rxembed as rx
import xyzrender
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")
tmp = tempfile.TemporaryDirectory()

## 1 · A pair of linkage isomers

Square-planar Ni(II) complexes of an O,N/O,C chelate, in pairs: the anionic donor is the amidate N in
one and the C of the carbanion in the other. Same formula, same phosphine backbone, different atom on
the metal. Three backbones, so the pairing is not an artefact of one ligand set.

Both are written with dative bonds (`->[Ni+2]<-`), and each enumerates its own distinct arrangements.
`geom_check.check` is the acceptance test: a sensible M-donor set is not sufficient on its own, since the
periphery can still be wrong and only the gate sees that.


In [ ]:
PAIRS = {
    "PEt2 diphosphine": (
        "CC[P]1(CC)CC[P](CC)(CC)->[Ni+2]<-12<-[O-]C(=O)C(c1ccccc1)[N-]->2c1ccccc1",
        "CC[P]1(CC)CC[P](CC)(CC)->[Ni+2]<-12<-[O-]C(=O)N(c1ccccc1)[CH-]->2c1ccccc1",
    ),
    "thiosemicarbazone": (
        "C[N]1(C)NC(N)=[S]->[Ni+2]<-12<-[O-]C(=O)C(c1ccccc1)[N-]->2c1ccccc1",
        "C[N]1(C)NC(N)=[S]->[Ni+2]<-12<-[O-]C(=O)N(c1ccccc1)[CH-]->2c1ccccc1",
    ),
    "dppe amidate": (
        "O=C1[O-]->[Ni+2]2(<-[N-](c3ccccc3)C1c1ccccc1)<-[P](CC[P]->2(c1ccccc1)c1ccccc1)(c1ccccc1)c1ccccc1",
        "O=C1[O-]->[Ni+2]2(<-[CH-](c3ccccc3)N1c1ccccc1)<-[P](CC[P]->2(c1ccccc1)c1ccccc1)(c1ccccc1)c1ccccc1",
    ),
}

for name, pair in PAIRS.items():
    for donor, smi in zip(("N-bound", "C-bound"), pair):
        isos = rx.metal(smi, "square_planar")
        ens = rx.embed(isos[0], n=3).minimize()
        clean = sum(report.ok() for report in ens.check().values())
        print(f"  {name:18s} {donor:8s} {len(isos)} isomer(s)  {ens.n} confs  {clean}/{ens.n} gate-clean")

Every donor distance and the coordination shape come back on each pose. The M-donor set is worth reading
back explicitly, since it is what the whole embed was arranged around.


In [ ]:
SMI = PAIRS["PEt2 diphosphine"][0]
iso = rx.metal(SMI, "square_planar")[0]
print(iso.summary())

ens = rx.embed(iso, n=6).minimize()
for report in ens.check().values():
    report.assert_ok()
for d in iso.donors:
    sym = f"{ens.mol.GetAtomWithIdx(d).GetSymbol()}{d}"
    print(f"  Ni-{sym:5s} {ens.measure((iso.metal, d))['mean']:.2f} A")
xyzrender.render(ens.mol, no_hy=True, bo=False)

## 2 · Searching it

`embed` seeds; `mc` searches. The chain is the same one an organic molecule gets, with
`preset='transition_metal'` so the search knows not to torsion-drive through the coordination sphere.
`prune` then dedups what came back.


In [ ]:
seeded = ens.n  # mc returns the same Ensemble, so read the seed count before searching
searched = ens.mc(preset="transition_metal").minimize()
before = searched.n
searched.prune()
for report in searched.check().values():
    report.assert_ok()
print(f"embed {seeded} -> mc {before} -> prune {searched.n} conformers")

xyzrender.render(searched.align().mol, no_hy=True, bo=False)

In [ ]:
searched.landscape()  # kept representatives solid, pruned-away duplicates faded

## 3 · Both hands of a racemate

The amidate backbone carries an undefined stereocentre, so the SMILES names a racemate rather than one
compound. `rx.embed(..., metal=)` returns an `EnsembleSet` with a candidate per hand, each tagged and each
carrying its own constraints through the whole chain.

The two are never pruned against each other: they are distinct species, and a force-field total is not
comparable across them. `best()` says so rather than ranking on a number that does not mean anything.


In [ ]:
racemate = rx.embed(SMI, metal="square_planar", n=4).minimize().prune()

for e in racemate:
    for report in e.check().values():
        report.assert_ok()
    print(f"  stereo {e.tag['stereo']}: {e.n} gate-clean conformers")

paths = racemate.dump(f"{tmp.name}/catalyst.xyz")  # one tagged .xyz per hand
print("dumped:", [p.split("/")[-1] for p in paths])

In [ ]:
try:
    racemate.best()
except ValueError as e:
    print("best() refused:", e)
# Ranking the hands needs real energies: racemate.score("gxtb").best() (xtb on $XTB_EXE).
# Enantiomers share a formula, so g-xTB totals do compare.

## 4 · A real metal TS: reacting core frozen, spectator ferrocene held

A bimetallic Mn/Fe transition state: H₂ activation at Mn (H–H breaking, Mn–H / substrate N–H, O–H forming),
with a spectator ferrocene. Freeze the reacting core while the periphery is searched, then check it with
the public frozen-core gate; both metals are restored. Here `ts_bonds=` is genuinely correct (real forming/breaking
bonds), labelled with their live lengths, note the ~0.86 Å H–H being cleaved.


In [ ]:
mn = "structures/mn-h2.xyz"
rc = [1, 5, 63, 64, 65, 66]  # reacting-core atoms (from a graphrc trajectory in practice)
bonds = [(5, 65), (64, 66), (1, 64), (65, 66), (63, 64)]  # forming / breaking bonds held during embed
ens = rx.embed(mn, fix=rc, n=6).mc(preset="ensemble").minimize()
total = ens.n
ens.filter("geometry", frozen=rc, reference=mn)
print(f"{total} conformers | frozen-core gate {ens.n}/{total} clean")
xyzrender.render(
    ens.mol,
    no_hy=True,
    ts_bonds=[(i + 1, j + 1) for i, j in bonds],
    labels=[f"{i + 1} {j + 1} d" for i, j in bonds],
)

## 5 · Seating a substrate, and switching its binding mode

Reach a substrate donor into the vacant site with `coordinate=`. A Pt(amine)Cl₂ has one open site; seat an
acetone carbonyl O there (a coordinative Pt–O bond, drawn dotted + labelled).


In [ ]:
isos = rx.metal("N->[Pt+2](<-[Cl-])<-[Cl-].CC(C)=O", "square_planar")  # 3 donors + acetone -> one vacant site
iso = isos.select(index=0)  # the cis isomer (both Cl cis)
O = next(a.GetIdx() for a in iso.mol.GetAtoms() if a.GetSymbol() == "O")
seated = rx.embed(iso, coordinate=O, n=3).mc(preset="ensemble").minimize().prune()
for report in seated.check().values():
    report.assert_ok()
print(f"acetone O seated: Pt-O = {seated.measure((iso.metal, O))['mean']:.2f} Å  ({len(seated)} confs)")
xyzrender.render(seated.mol, no_hy=True, labels=[f"{iso.metal + 1} {O + 1} d"])  # label the new Pt-O coordinative bond

The same substrate can bind the other way instead: not seated on the metal at all, but hydrogen-bonded to
the amine. One `contacts=` grip states it, and `Pt-O` reports which mode you got.

The embed biases a grip toward the window midpoint and restrained UFF keeps it inside the window. `mc` does
not model the contact itself, so `minimize` reapplies the constraint afterwards.


In [ ]:
N = next(d for d in iso.donors if iso.mol.GetAtomWithIdx(d).GetSymbol() == "N")
H = next(n.GetIdx() for n in iso.mol.GetAtomWithIdx(N).GetNeighbors() if n.GetAtomicNum() == 1)
grip = rx.Contact(distances={(min(H, O), max(H, O)): (1.7, 2.3)}, angles={(N, H, O): (150.0, 180.0)}, label="NH..O")

hb = rx.embed(iso, contacts=grip, n=4)  # no coordinate=: the acetone grips the amine, not the metal
print(
    f"seeded  : H-O {hb.measure((H, O))['mean']:.2f} A (window 1.7-2.3), "
    f"Pt-O {hb.measure((iso.metal, O))['mean']:.2f} A"
)
hb = hb.mc(preset="ensemble").minimize().prune()
for report in hb.check().values():
    report.assert_ok()
print(
    f"searched: H-O {hb.measure((H, O))['mean']:.2f} A, "
    f"N-H-O {hb.measure((N, H, O))['mean']:.0f} deg, {hb.n} gate-clean"
)

xyzrender.render(hb.mol, no_hy=True, nci_bonds=[(H + 1, O + 1)], labels=[f"{H + 1} {O + 1} d"])